# Model Training

Training two models:
- **MobileNetV2** — pretrained on ImageNet, fine-tuned for quality score regression
- **QualityCNN** — small custom CNN as baseline comparison

Plus per-issue logistic regression classifiers on the 18 features.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json, pickle, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from config import DATA_DIR, MODELS_DIR, IMG_SIZE, CNN_EPOCHS, CNN_BATCH_SIZE, CNN_LR, RANDOM_SEED
from ml.model import MobileNetV2Quality, QualityCNN
from ml.features import extract_features, ImageFeatures
from ml.synthetic_data import ISSUE_TYPES

MODELS_DIR.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_GEN = DATA_DIR / "generated"
print(f"Device: {DEVICE}, Epochs: {CNN_EPOCHS}, Batch: {CNN_BATCH_SIZE}")

## Dataset loader

In [ ]:
class QualityDataset(Dataset):
    def __init__(self, split):
        self.img_dir = DATA_GEN / split / "images"
        self.meta = pd.read_csv(DATA_GEN / split / "metadata.csv")
    
    def __len__(self): return len(self.meta)
    
    def __getitem__(self, i):
        row = self.meta.iloc[i]
        img = cv2.imread(str(self.img_dir / row['filename']))
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        t = torch.from_numpy((img[:,:,::-1].astype(np.float32) / 255.0 - 0.5) / 0.5).permute(2, 0, 1).contiguous()
        return t, torch.tensor(row['quality_score'], dtype=torch.float32)

train_ds = QualityDataset("train")
val_ds = QualityDataset("val")
test_ds = QualityDataset("test")
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

train_loader = DataLoader(train_ds, batch_size=CNN_BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=CNN_BATCH_SIZE, shuffle=False, num_workers=0)

## Training loop

In [ ]:
def train(model, train_loader, val_loader, epochs, lr, name="model"):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.SmoothL1Loss()
    
    history = []
    best_mae, best_state = float('inf'), None
    
    for ep in range(epochs):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            losses.append(loss.item())
        sched.step()
        
        # validate
        model.eval()
        preds, gts = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                p = model(xb.to(DEVICE)).cpu().numpy()
                preds.extend(np.clip(p, 0, 100))
                gts.extend(yb.numpy())
        
        mae = np.mean(np.abs(np.array(preds) - np.array(gts)))
        history.append({'epoch': ep+1, 'loss': np.mean(losses), 'val_mae': mae})
        
        if mae < best_mae:
            best_mae = mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        
        if (ep+1) % 5 == 0 or ep == 0:
            print(f"[{name}] ep {ep+1}/{epochs} loss={np.mean(losses):.4f} val_mae={mae:.2f}")
    
    model.load_state_dict(best_state)
    print(f"[{name}] best val MAE: {best_mae:.2f}")
    return model, history

## Train MobileNetV2

Phase 1: frozen backbone (warm up the head), Phase 2: unfreeze and fine-tune.

In [ ]:
mobilenet = MobileNetV2Quality(pretrained=True, freeze_backbone=True)

print("Phase 1: frozen backbone")
mobilenet, hist1 = train(mobilenet, train_loader, val_loader, epochs=5, lr=CNN_LR, name="MobileNet-frozen")

print("\nPhase 2: fine-tuning")
mobilenet.unfreeze_backbone(from_layer=-5)
mobilenet, hist2 = train(mobilenet, train_loader, val_loader, epochs=CNN_EPOCHS-5, lr=CNN_LR*0.1, name="MobileNet-ft")

torch.save(mobilenet.state_dict(), MODELS_DIR / "mobilenet_quality.pt")
print(f"Saved to {MODELS_DIR / 'mobilenet_quality.pt'}")

full_hist = hist1 + [{'epoch': h['epoch']+5, **{k:v for k,v in h.items() if k!='epoch'}} for h in hist2]

## Train QualityCNN (baseline)

In [ ]:
cnn = QualityCNN()
cnn, cnn_hist = train(cnn, train_loader, val_loader, epochs=CNN_EPOCHS, lr=CNN_LR, name="QualityCNN")
torch.save(cnn.state_dict(), MODELS_DIR / "cnn_quality.pt")
print(f"Saved to {MODELS_DIR / 'cnn_quality.pt'}")

## Training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for ax, hist, title in [(ax1, full_hist, "MobileNetV2"), (ax2, cnn_hist, "QualityCNN")]:
    eps = [h['epoch'] for h in hist]
    ax.plot(eps, [h['loss'] for h in hist], 'b-', label='Loss', alpha=0.8)
    ax.plot(eps, [h['val_mae'] for h in hist], 'r-', label='Val MAE', alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.3)

if len(full_hist) > 5:
    ax1.axvline(x=5, color='gray', ls='--', alpha=0.5, label='unfreeze')

plt.tight_layout()
plt.savefig(str(DATA_GEN / "training_curves.png"), dpi=150, bbox_inches='tight')
plt.show()

## Train per-issue classifiers

One LogisticRegression per issue type, trained on the 18 features.

In [ ]:
def build_features(split, max_n=None):
    meta = pd.read_csv(DATA_GEN / split / "metadata.csv")
    img_dir = DATA_GEN / split / "images"
    if max_n and len(meta) > max_n:
        meta = meta.sample(max_n, random_state=RANDOM_SEED).reset_index(drop=True)
    
    X, Y = [], {t: [] for t in ISSUE_TYPES}
    for _, row in tqdm(meta.iterrows(), total=len(meta), desc=split):
        img = cv2.imread(str(img_dir / row['filename']))
        if img is None: continue
        X.append(extract_features(img).to_vector())
        for t in ISSUE_TYPES:
            col = f'issue_{t}'
            Y[t].append(int(row[col]) if col in row else 0)
    return np.array(X), {t: np.array(v) for t, v in Y.items()}

X_train, y_train = build_features("train", max_n=5000)
X_val, y_val = build_features("val")

# combine for final training
X_all = np.vstack([X_train, X_val])
y_all = {t: np.concatenate([y_train[t], y_val[t]]) for t in ISSUE_TYPES}
print(f"Total training features: {X_all.shape}")

In [ ]:
classifiers = {}
results = {}
feat_names = ImageFeatures.names()

for issue in ISSUE_TYPES:
    y = y_all[issue]
    if len(np.unique(y)) < 2:
        print(f"[{issue}] skipping — single class")
        continue
    
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X_all)
    
    clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED)
    clf.fit(Xs, y)
    
    # quick check on training data
    pred = clf.predict(Xs)
    acc = accuracy_score(y, pred)
    f1 = f1_score(y, pred, zero_division=0)
    
    top3 = sorted(zip(feat_names, clf.coef_[0].tolist()), key=lambda t: -abs(t[1]))[:3]
    print(f"[{issue}] acc={acc:.3f} f1={f1:.3f} top: {[(n, round(w,2)) for n,w in top3]}")
    
    classifiers[issue] = {"scaler": scaler, "model": clf}
    results[issue] = {"accuracy": acc, "f1": f1, "top_features": top3}

with open(MODELS_DIR / "issue_classifiers.pkl", "wb") as f:
    pickle.dump(classifiers, f)
with open(MODELS_DIR / "feature_names.json", "w") as f:
    json.dump(feat_names, f)
print(f"\nSaved classifiers and feature names to {MODELS_DIR}")

## Save training report

In [ ]:
report = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "device": DEVICE,
    "dataset": {"train": len(train_ds), "val": len(val_ds), "test": len(test_ds)},
    "mobilenet_history": full_hist,
    "cnn_history": cnn_hist,
    "classifiers": {k: {"accuracy": v["accuracy"], "f1": v["f1"]} for k, v in results.items()},
}
with open(MODELS_DIR / "training_report.json", "w") as f:
    json.dump(report, f, indent=2)
print("Training done.")